# Week 2 Final Exercise
In this solution we are going to create a Gradio UI that will allow the user to as use the technical answerer to answer questions in a chat with history. 

In [2]:
# imports

import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [3]:
# Initialization

load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
GPT_MODEL = "gpt-4.1-mini"
openai = OpenAI()

OLLAMA_MODEL = "llama3.2"
ollama = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')


OpenAI API Key exists and begins sk-proj-


In [ ]:
# prompts

question = ""
system_prompt = "You are a helpful technical tutor who answers questions about python code, \
    software engineering, data science and LLMs.\
    You do not answer questions about any other programming language except phyton. \
    If you are asked about any other language you should say something like \"I'm sorry I only understand python\" \
    You can answer questions about SQL as well."
user_prompt = "Please give a detailed explanation to the following question: " + question

# messages

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt}
]

In [28]:
def count_characters(message):
    count = len(message)
    chars = f"*Counting characters*. Total = {count}"
    return chars

count_function = {
    "name": "count_characters",
    "description": "Counts the number of characters in the input string.",
    "parameters": {
        "type": "object",
        "properties": {
            "message": {
                "type": "string",
                "description": "The chat history and messages.",
            },
        },
        "required": ["message"],
        "additionalProperties": False
    }
}

tools = [{"type": "function", "function": count_function}]

In [29]:
print(count_characters("How long is thisffff?"))


*Counting characters*. Total = 21


In [26]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "count_characters":
            counter = count_characters(message)
            responses.append({
                "role": "tool",
                "content": counter,
                "tool_call_id": tool_call.id
            })
    return responses

In [48]:
def use_chosen_model(model, messages, stream=False):
    response = ""
    if model=="GPT":
        response = openai.chat.completions.create(model=GPT_MODEL, messages=messages, stream=stream)
    elif model=="OLLAMA":
        response = ollama.chat.completions.create(model=OLLAMA_MODEL, messages=messages, stream=stream)
    else:
        raise ValueError("Unknown model")

    return response

def chat(history, model):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_prompt}] + history
    
    ## No Streaming ---------------------------------------------------
    #response = use_chosen_model(model, messages)
    
    ##while response.choices[0].finish_reason=="tool_calls":
    ##    message = response.choices[0].message
    ##    responses = handle_tool_calls(message)
    ##    messages.append(message)
    ##    messages.extend(responses)
    ##    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    
    #reply = response.choices[0].message.content
    #history += [{"role":"assistant", "content":reply}]    
    
    #return history
    ## No Streaming ---------------------------------------------------

    ## Stream ---------------------------------------------------------
    #if model=="GPT":
    #    stream = openai.chat.completions.create(model=GPT_MODEL, messages=messages, stream=True)
    #elif model=="OLLAMA":
    #    stream = ollama.chat.completions.create(model=OLLAMA_MODEL, messages=messages, stream=True)
    #else:
    #    raise ValueError("Unknown model")
    stream = use_chosen_model(model, messages, True)
    
    reply = ""
    for chunk in stream:
        token = chunk.choices[0].delta.content or ""
        reply += token
        
    current_history = history + [{"role":"assistant", "content":reply}] 
    yield current_history
    
    ## Stream ---------------------------------------------------------
    

In [49]:
# Callbacks (along with the chat() function above)

def put_message_in_chatbot(message, history):
        return "", history + [{"role":"user", "content":message}]

# UI definition

with gr.Blocks() as ui:
    with gr.Row():
        model_selector = gr.Dropdown(["GPT", "OLLAMA"], label="Select model", value="GPT")
    with gr.Row():
        chatbot = gr.Chatbot(height=200, type="messages")
    with gr.Row():
        message = gr.Textbox(label="Chat with our AI Assistant:")

# Hooking up events to callbacks

    message.submit(put_message_in_chatbot, inputs=[message, chatbot], outputs=[message, chatbot]).then(
        chat, inputs=[chatbot, model_selector], outputs=[chatbot]
    )

ui.launch()

* Running on local URL:  http://127.0.0.1:7876
* To create a public link, set `share=True` in `launch()`.
